# Temas Tratados en el Trabajo Práctico 6

* Modelado de problemas en espacios de estado.

* Algoritmos de planificación hacia adelante y hacia atrás.

* Representación y solución de problemas descritos en lenguaje STRIPS.

* Algoritmo GRAPHPLAN.

* Planificación con restricciones de tiempo y recursos.

* Caminos críticos y tiempos de relajación.

## Ejercicios Teóricos

1. ¿En qué tipo de algoritmos se basa un planificador para encontrar el mejor camino a un estado solución?

### Resolución

Un planificador se basa en **algoritmos de búsqueda en espacios de estados**. El problema de planificación se representa mediante:

- un **estado inicial**;
- un conjunto de **acciones** que permiten pasar de un estado a otro;
- un **objetivo** que debe cumplirse;
- un costo asociado al plan.

Cada estado constituye un nodo del espacio de búsqueda y cada acción aplicable representa una transición:

$$
s_0 \xrightarrow{a_1} s_1 \xrightarrow{a_2} \cdots
\xrightarrow{a_n} s_n
$$

El planificador debe encontrar una secuencia de acciones:

$$
\pi=\langle a_1,a_2,\ldots,a_n\rangle
$$

tal que el estado final $s_n$ satisfaga el objetivo $g$:

$$
g\subseteq s_n
$$

La búsqueda puede realizarse en dos direcciones:

- **Hacia adelante o por progresión:** comienza en el estado inicial y aplica sucesivamente las acciones habilitadas hasta encontrar un estado que satisfaga el objetivo.
- **Hacia atrás o por regresión:** comienza en el objetivo y busca qué acciones podrían alcanzarlo, reemplazando progresivamente los objetivos por las precondiciones de esas acciones hasta llegar al estado inicial.

Para explorar el espacio pueden utilizarse algoritmos de búsqueda **no informada**, como búsqueda en anchura o costo uniforme, o algoritmos de búsqueda **informada**, como A*, que emplean una función heurística para priorizar los estados más prometedores:

$$
f(n)=g(n)+h(n)
$$

donde:

- $g(n)$ es el costo acumulado desde el estado inicial;
- $h(n)$ es una estimación del costo restante hasta el objetivo.

Si todas las acciones tienen costo unitario, el costo de un plan coincide con su cantidad de acciones:

$$
Costo(\pi)=|\pi|
$$

Por lo tanto, encontrar el mejor camino significa encontrar un plan válido de costo mínimo. En el caso de costo unitario, esto equivale a encontrar el plan que alcance el objetivo utilizando la menor cantidad de acciones.


2. ¿Qué tres elementos se encuentran dentro de una acción formulada en lenguaje STRIPS? Describa brevemente qué función cumple cada uno.

### Resolución

En STRIPS, una acción se identifica mediante un **nombre** y, cuando corresponde, una lista de parámetros. Su comportamiento se describe mediante tres componentes principales:

#### 1. Precondiciones

Las precondiciones indican qué hechos deben ser verdaderos antes de ejecutar la acción. Una acción $a$ puede aplicarse en un estado $s$ solamente cuando todas sus precondiciones se encuentran satisfechas:

$$
Precond(a)\subseteq s
$$

#### 2. Lista de agregados

La lista $Add(a)$ contiene los hechos que pasan a ser verdaderos después de ejecutar la acción.

#### 3. Lista de eliminados

La lista $Del(a)$ contiene los hechos que dejan de ser verdaderos como consecuencia de la acción.

El nuevo estado se calcula conservando los hechos que no cambian, eliminando los indicados por $Del(a)$ y agregando los indicados por $Add(a)$:

$$
s'=(s\setminus Del(a))\cup Add(a)
$$

Por ejemplo, la acción de tomar una pieza puede representarse como:

$$
Acción(Tomar(P1))
$$

$$
Precond=
\{
En(P1,Mesa),\ PinzaLibre
\}
$$

$$
Add=
\{
Sujeta(P1)
\}
$$

$$
Del=
\{
En(P1,Mesa),\ PinzaLibre
\}
$$

Si el estado inicial es:

$$
s=
\{
En(P1,Mesa),\ PinzaLibre,\ SoporteLibre
\}
$$

al aplicar la acción se obtiene:

$$
s'=
\{
Sujeta(P1),\ SoporteLibre
\}
$$

El hecho $SoporteLibre$ permanece verdadero porque la acción no lo incluye en su lista de eliminados. De esta manera, STRIPS modifica únicamente los hechos indicados explícitamente en los efectos de la acción.

3. Describa las ventajas y desventajas de desarrollar un algoritmo de planificación hacia adelante y hacia atrás en el espacio de estados.


### Resolución

Los problemas de planificación pueden resolverse mediante búsqueda hacia adelante, denominada **progresión**, o mediante búsqueda hacia atrás, denominada **regresión**. Ambas estrategias buscan una secuencia de acciones válida, pero recorren el problema en sentidos opuestos.

#### Planificación hacia adelante o por progresión

La progresión comienza en el estado inicial $s_0$. En cada estado identifica las acciones cuyas precondiciones se cumplen y genera los estados sucesores:

$$
s'=(s\setminus Del(a))\cup Add(a)
$$

La búsqueda finaliza cuando encuentra un estado que satisface el objetivo:

$$
g\subseteq s
$$

Su principal ventaja es que resulta sencilla de implementar, porque en cada estado se conoce directamente qué acciones son aplicables y cómo calcular sus sucesores. Además, puede utilizar algoritmos de búsqueda conocidos, como búsqueda en anchura, costo uniforme o A*.

Su principal desventaja es que puede presentar un factor de ramificación elevado. El planificador considera todas las acciones aplicables, incluso aquellas que no contribuyen al objetivo. Esto puede producir una gran cantidad de estados alternativos y repetidos, provocando una explosión del espacio de búsqueda.

Por ejemplo, desde un estado inicial podrían generarse varias ramas:

$$
s_0
\begin{cases}
\xrightarrow{a_1}s_1^1\\
\xrightarrow{a_2}s_1^2\\
\xrightarrow{a_3}s_1^3
\end{cases}
$$

aunque solo una de ellas resulte útil para alcanzar la meta.

#### Planificación hacia atrás o por regresión

La regresión comienza con el conjunto de objetivos $G$ y busca acciones cuyos efectos permitan alcanzarlos.

Una acción $a$ es **relevante** si consigue al menos uno de los objetivos:

$$
Add(a)\cap G\neq\varnothing
$$

Además, debe ser **consistente**, es decir, no debe eliminar otro objetivo que también se necesite:

$$
Del(a)\cap G=\varnothing
$$

Una vez seleccionada una acción, los objetivos que esta consigue se reemplazan por sus precondiciones:

$$
Regresar(G,a)=
(G\setminus Add(a))\cup Precond(a)
$$

La búsqueda continúa hasta que el estado inicial satisface todos los subobjetivos regresados.

Su principal ventaja es que considera solamente acciones relacionadas con el objetivo, evitando muchas acciones irrelevantes y reduciendo en algunos problemas la cantidad de ramas exploradas.

Su desventaja es que resulta más difícil de implementar. No siempre es evidente cuáles son los estados predecesores de un objetivo, y se deben manipular conjuntos de subobjetivos parcialmente especificados. También es necesario controlar que las acciones elegidas no eliminen otros hechos deseados y decidir entre varias acciones cuando un mismo objetivo puede alcanzarse de diferentes maneras.

En síntesis:

| Característica | Progresión | Regresión |
|---|---|---|
| Punto de partida | Estado inicial | Objetivo |
| Dirección | $s_0\rightarrow g$ | $g\rightarrow s_0$ |
| Selección de acciones | Acciones aplicables | Acciones relevantes y consistentes |
| Elementos explorados | Estados completos | Conjuntos de subobjetivos |
| Ventaja principal | Implementación más directa | Evita muchas acciones irrelevantes |
| Desventaja principal | Alto factor de ramificación | Generación de predecesores más compleja |

Por lo tanto, la progresión pregunta:

> ¿Qué acciones puedo ejecutar desde el estado actual?

Mientras que la regresión pregunta:

> ¿Qué acción debería ejecutarse antes para alcanzar el objetivo actual?

4. Considere el problema de ponerse uno mismo zapatos y medias. Aplique GRAPHPLAN a este problema y muestre la solución obtenida. Muestre el plan de orden parcial que es solución e indique cuántas linealizaciones diferentes existen para el plan de orden parcial.

5. Se requiere ensamblar una máquina cuyas piezas están identificadas con las letras A, B, C, D y E. El tiempo que se tarda en ensamblar cada pieza es:

* A: 2 semanas

* B: 1 semana

* C: 4 semanas

* D: 3 semanas

* E: 5 semanas

    El orden de ensamblaje de cada pieza requiere que:

* A esté realizado antes que C

* B esté realizado antes que C

* B esté realizado antes que D

* C esté realizado antes que E

* D esté realizado antes que E

    Con esta información:

        5.1 Arme el Plan de Orden Parcial.

        5.2 Encuentre el Camino Crítico.

        5.3 Encuentre los tiempos de relajación.

        5.4 Dibuje un diagrama temporal indicando las tareas y los tiempos de relajación encontrados.

## Ejercicios de Implementación

> Recuerde adjuntar en la presentación el prompt inicial que ha utilizado para cada ejercicio de implementación y si considera que le dio la información completa para resolver el ejercicio o qué cambios adicionales tuvo que pedir de manera iterativa.

6. Suponga que tiene un robot de oficina capaz de moverse y tomar y depositar objetos. El robot solo puede tener un objeto a la vez, pero puede conseguir una *caja* en la que depositar varios objetos. Suponga que programa al robot para *ir a la tienda* a comprarle un *café* y en el camino de vuelta tome una *carta* del *buzón* de la oficina para para que se la traiga junto con el café. Describa en lenguaje STRIPS:

        6.1 El dominio del robot (nombre, predicados y acciones que puede hacer el robot).

        6.2 El problema que se quiere resolver (estado inicial, estado objetivo y objetos del mundo representados).

        6.3 Introduzca el código desarrollado en los puntos anteriores en el [planificador online](http://lcas.lincoln.ac.uk/fast-downward/) y obtenga el plan de acción que tomará el robot para cumplir lo solicitado.


# Bibliografía

[Russell, S. & Norvig, P. (2004) _Inteligencia Artificial: Un Enfoque Moderno_. Pearson Educación S.A. (2a Ed.) Madrid, España](https://www.academia.edu/8241613/Inteligencia_Aritificial_Un_Enfoque_Moderno_2da_Edici%C3%B3n_Stuart_J_Russell_y_Peter_Norvig)

[Poole, D. & Mackworth, A. (2023) _Artificial Intelligence: Foundations of Computational Agents_. Cambridge University Press (3a Ed.) Vancouver, Canada](https://artint.info/3e/html/ArtInt3e.html)